<!-- nav -->
[← 2 · Choosing what to apply: candidates from a frame stream](02_Choosing_What_To_Apply.ipynb) · [Home](../../README.md) · [TypeScript tutorial →](../../typescript5/tutorials/README.md)

# 3 · Running again: reruns and diffs, queries over the certificates

**The problem.** The shelf changes: an item is added, another removed. Run again, the rewrite should make the same
decisions where the same transform applies to the same frame, and ask only about what is new. And a reviewer wants to
know what changed between two runs: what was added, what was removed, what was decided differently.

mbse-transforms has no call for either. A transform certificate records its transform, its parameter assignments and
each sub-certificate's frame ("ofc those parameters must be recorded in the transform certificate"), and certificates
are mbse-schemas data. So a rerun's reuse is a query over the certificates' store (3.2), and a diff the same query
over two histories, with mbse-schemas' `Comparison` (3.3). Calls of mbse-transforms' own for them would be bespoke
code reading a model, which the user ruled out: "No model should need bespoke schema-aware code to read or write. Use
mbse-schemas built-in serialization support."

The shelf, its prototype and `label` are case study 1's; `run` is a session as in case study 2, with each frame's
case decided by a function you give it.

In [1]:
from mbse.Schemas.Framework import Builders, Comparison, Plain, Proxies, References
from mbse.Transforms import Transforms
from toolkit import assignments_of, bear, frames, items, model, resolve, shown


def label(source, sync, target, assignments):
    """Labels the sync's item with its name, in the case its parameter assignment gives."""
    item = sync.i
    name = str(item.name)
    Proxies.store_of(item).shelf.Item(item, lambda i: i.label(name.upper() if assignments["case"] == "upper" else name.lower()))


transforms = model()  # the transforms' store
transforms.transaction(Transforms.install)
transforms.transaction(lambda t: Transforms.define(t, "Label", "label", t.shelf.Unlabelled))
procedures = Transforms.Procedures()
procedures.register("label", label)
certs = Builders.Store()  # the certificates' store
certs.transaction(Transforms.install)


def run(data, history, decide):
    """Your run over the data's frames of shelf.Unlabelled: for each frame, `decide(frame)` gives its case and who
    decided it; the candidate of Label with that case is applied, its frame the sync, and certified into the history
    named `history`."""
    found = frames(data, "shelf.Unlabelled")
    with transforms.transaction(), data.transaction():
        Label = transforms.singleton("Transforms.TheCatalog").transforms.at("Label").transform
        for candidates in Transforms.Session([Label], (bear(data, "shelf.Unlabelled", f) for f in found)):
            case, by = decide(candidates[0].frame)
            (taken,) = [c for c in candidates if c.assignments["case"] == case]
            Transforms.apply(Label, taken.assignments, procedures, sync=taken.frame)
            Transforms.certify(certs, Label, taken.assignments, history, by, sync=taken.frame)


def chosen(choices):
    """You deciding each item's case yourself, by its name."""
    return lambda frame: (choices[str(frame.i.name)], "caller")


def key_of(frame):
    """The key on the shelf of the item a sub-certificate's frame binds, read from the frame's plain data."""
    found = []
    frame.as_object(lambda o: found.append(Plain.ToPlain(certs)(o.as_schema(), o)) or o)
    return found[0]["journal"]["symbols"]["i"]["reference"]["path"]["next"]["entry"]["properties"]["key"]


def told(certificate):
    """A transform certificate, read within a transaction: the key of the item its frame binds, and its parameter
    assignments."""
    return f"{key_of(next(iter(certificate.frames)).frame.frame)} {shown(assignments_of(certificate))}"


def history_of(name):
    """Prints each transform certificate of the history named `name`, and who decided it."""
    with certs.transaction():
        history = certs.singleton("Transforms.TheHistories").histories.at(name).history
        for entry, decided in zip(history.steps, history.decisions):
            print(f"{told(entry.step)}, decided by {decided.by}")

<a id="T03.1"></a>
## 03.1 · A first run, recorded

A shelf of bolt and nut, each case decided by you, certified into the history `first`:

In [2]:
first = model("bolt", "nut")
run(first, "first", chosen({"bolt": "upper", "nut": "lower"}))
history_of("first")

bolt case=upper, decided by caller
nut case=lower, decided by caller


<a id="T03.2"></a>
## 03.2 · A rerun reuses what the certificates recorded

To reuse a decision, query the certificates' store for the transform certificate, in the earlier history, of the
same transform on the same frame: a `map` whose spec names the history, its `steps`' transform certificate whose
`transform` equals the transform's object reference, and its `frames`' sub-certificate whose `frame` equals the frame.
A spec matches a value by equality of the whole value, so the transform's reference and the frame's bindings are made
values of the certificates' store first, written as plain data and read back (`carried`). A frame binds by object
reference, so a shelf remade with the same owners and keys is queried by its own frames.

Where the query finds a certificate, read the parameter assignments it recorded, within a transaction, and apply and
certify with them, decided "reused"; where it finds none, decide as for a new frame.

In [3]:
def carried(own, x):
    """The object reference of `x` (an object, or a frame's bindings), made in the store `own`, as a value of the
    certificates' store: written as plain data and read back."""
    reference = References.of(own, x)
    return Plain.FromPlain(certs)(certs.registered(reference.as_schema().name),
                                  Plain.ToPlain(own)(reference.as_schema(), reference))


def recorded(history, transform, frame):
    """The query: the frames binding each transform certificate `c`, in the history named `history`, of the transform
    `transform`, and its sub-certificate `f` holding the frame `frame`."""
    T = certs.Transforms
    return certs.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
        .symbols({"d": T.Histories, "h": T.HistoryCertificate, "c": T.TransformCertificate, "f": T.FrameCertificate})
        .spec([
            lambda _: _.d.histories({history: lambda e: e.history(_.h)}),
            lambda _: _.h.steps([lambda e: e.step(_.c)]),
            lambda _: _.c.transform(transform).frames([lambda e: e.frame(_.f)]),
            lambda _: _.f.frame(frame),
        ])]))]))


with transforms.transaction():
    label_ref = carried(transforms, transforms.singleton("Transforms.TheCatalog").transforms.at("Label").transform)


def reusing(history, data, otherwise):
    """Deciding as the history named `history` recorded, for a frame of `data`'s: the parameter assignments the query
    finds, decided "reused"; where it finds none, `otherwise`'s decision."""
    def decide(frame):
        found = recorded(history, label_ref, carried(data, {"i": frame.i}))
        if not found:  # a frame the earlier history never saw
            return otherwise(frame)
        (given,) = found
        with certs.transaction():  # the certificate the frame given back binds, read within a transaction
            return assignments_of(resolve(certs, given["c"]))["case"], "reused"
    return decide


again = model("bolt", "nut", "washer")  # remade, with an item the first run never saw
run(again, "second", reusing("first", again, chosen({"washer": "upper"})))
history_of("second")
items(again)

bolt case=upper, decided by reused
nut case=lower, decided by reused
washer case=upper, decided by caller
bolt BOLT
nut nut
washer WASHER


<a id="T03.3"></a>
## 03.3 · A diff: two histories compared

A diff lists each history's certificates by the same kind of query, looks each up in the other history by the query
of 3.2, and compares each pair it finds with mbse-schemas' `Comparison.OfObject`: a certificate with none in the other
history was removed (an orphan) or added, and a pair not equal recorded other parameter assignments. Each history's
certificates are listed with their transform and frame, as values of the certificates' store, to query the other by.

In [4]:
def certified(history):
    """The query a diff starts from: the frames binding each transform certificate `c` in the history named `history`
    and each of its sub-certificates `f`."""
    T = certs.Transforms
    return certs.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
        .symbols({"d": T.Histories, "h": T.HistoryCertificate, "c": T.TransformCertificate, "f": T.FrameCertificate})
        .spec([
            lambda _: _.d.histories({history: lambda e: e.history(_.h)}),
            lambda _: _.h.steps([lambda e: e.step(_.c)]),
            lambda _: _.c.frames([lambda e: e.frame(_.f)]),
        ])]))]))


def held(member):
    """The value object a certificate's property holds, as a value of the certificates' store: written as plain data
    and read back."""
    found = []
    member.as_object(lambda o: found.append(Plain.FromPlain(certs)(o.as_schema(), Plain.ToPlain(certs)(o.as_schema(), o))) or o)
    return found[0]


def same(a, b):
    """Whether two transform certificates are equal as values (mbse-schemas' `Comparison`): their transform, prototype
    and parameter assignments."""
    return Comparison.OfObject(a.as_schema(), a).compare(Comparison.OfObject(b.as_schema(), b)) == 0


def diff(a, b):
    """What the history named `b` removes from the one named `a`, what it adds, and what it changes, each by object
    reference."""
    listed = {}
    for name in (a, b):
        found = certified(name)
        with certs.transaction():
            listed[name] = [(given["c"], held(resolve(certs, given["c"]).transform), held(resolve(certs, given["f"]).frame))
                            for given in found]
    removed = [c for c, transform, frame in listed[a] if not recorded(b, transform, frame)]
    added = [c for c, transform, frame in listed[b] if not recorded(a, transform, frame)]
    pairs = [(c, given["c"]) for c, transform, frame in listed[a] for given in recorded(b, transform, frame)]
    with certs.transaction():
        changed = [(c, d) for c, d in pairs if not same(resolve(certs, c), resolve(certs, d))]
    return removed, added, changed


before, after = model("bolt", "nut", "washer"), model("bolt", "nut", "screw")  # washer removed, screw added
run(before, "before", chosen({"bolt": "upper", "nut": "upper", "washer": "lower"}))
run(after, "after", chosen({"bolt": "upper", "nut": "lower", "screw": "lower"}))  # nut decided otherwise
removed, added, changed = diff("before", "after")
with certs.transaction():
    print("removed:", "; ".join(told(resolve(certs, c)) for c in removed))
    print("added:", "; ".join(told(resolve(certs, c)) for c in added))
    print("changed:", "; ".join(f"{told(resolve(certs, c))} -> {told(resolve(certs, d))}" for c, d in changed))
print("a rerun that reused, against its first run:", [len(found) for found in diff("first", "second")])

removed: washer case=lower
added: screw case=lower
changed: nut case=upper -> nut case=lower
a rerun that reused, against its first run: [0, 1, 0]


Bolt's certificates are equal in both histories, so it is in neither list; the rerun of 3.2 added only washer,
and changed nothing. Reruns and diffs need nothing more of mbse-transforms than the certificates it writes.

<!-- nav -->
[← 2 · Choosing what to apply: candidates from a frame stream](02_Choosing_What_To_Apply.ipynb) · [Home](../../README.md) · [TypeScript tutorial →](../../typescript5/tutorials/README.md)